# E2.8 · Auditability of autonomous action

**Function E — Governance, Risk, Compliance & the CISO Office → The Regulatory & Compliance Lead**  ·  *Security of AI*

---

**Risk.** No trail showing under whose authority the agent acted.

**Control.** The delegation chain *is* the audit trail.

**This lab.** Build the audit trail out of the delegation chain itself.

| | |
|---|---|
| Open-source tooling | Keycloak |
| Open-weight models | — |

> Runs anywhere: standard library only, no network, no API key. Where a lesson names a real tool you would deploy (Falco, OPA, SPIRE, Keycloak), the notebook models the *decision* that tool makes, so the lesson still lands on a machine that cannot pull containers.

In [ ]:
# --- Cyber Commons bootstrap -------------------------------------------------
# Puts the lab library on the path. Works from a clone, from the repo root, and
# on Kaggle. Standard library only — nothing to install, no network required.
import sys, os, subprocess
from pathlib import Path

def _find_labs():
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / "labs" / "cybercommons" / "__init__.py").is_file():
            return base / "labs"
    # Kaggle kernels start in /kaggle/working with the repo absent. If the
    # kernel has internet enabled we clone it; if not, this raises and the
    # message tells you to attach the repo as a dataset instead.
    dest = Path("/kaggle/working/cyber-commons")
    if not dest.exists():
        subprocess.run(["git", "clone", "--depth", "1", "--branch", "claude/vulnbench-setup-scheduling-81aqov",
                        "https://github.com/spbreed/cyber-commons", str(dest)], check=True)
    return dest / "labs"

sys.path.insert(0, str(_find_labs()))
import cybercommons
print(cybercommons.banner("E2.8"))

Auditability of autonomous action reduces to one question: can you produce, for any single action, who caused it and what they were allowed to do?

In [ ]:
from cybercommons import identity, ir

alice = identity.mint("alice")
patch = identity.exchange(alice, "patch-agent", {"repo:read", "repo:write"})

print("auditable record of one action:")
print(f"   action        write_file /etc/app.conf")
print(f"   acting id     {patch.actor}")
print(f"   on behalf of  {patch.sub}")
print(f"   chain         {' → '.join(patch.chain())}")
print(f"   scopes held   {sorted(patch.scopes)}")
print(f"   token fp      {patch.fingerprint()}")

Now the same action under impersonation — the record an auditor would actually receive from most deployments today.

In [ ]:
bad = identity.impersonate("alice", "patch-agent", {"repo:write"})
print(f"   acting id     {bad.actor}   ← the human")
print(f"   chain         {' → '.join(bad.chain())}")
print("   the agent does not appear. The record is complete, consistent, and false.")

ok, missing = ir.Replay(["prompt"], ["tool result"], "glm-4.6@2025-11", 42).replayable()
print(f"\nreplayable: {ok}  (auditability = attribution + replay, not one of them)")

### Expect

The delegated record names the acting identity, the principal, the chain, the scopes and a fingerprint. The impersonated record shows only alice, and the replay check passes for a fully instrumented run.

### Your turn

Pick one production agent action from last week and try to produce this record. Whatever field you cannot fill is your auditability gap, stated precisely.

---

[All lessons](https://github.com/spbreed/cyber-commons/tree/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks) · [Lesson page](https://spbreed.github.io/cyber-commons/lessons/E2.8.html) · [Lab library](https://github.com/spbreed/cyber-commons/tree/claude/vulnbench-setup-scheduling-81aqov/labs/cybercommons)

*Cyber Commons — a free, open commons for Cyber AI.*